In [1]:
from google.colab import drive

drive.mount("/content/gdrive")

Mounted at /content/gdrive


LOC-01 — Initialize

In [2]:
import os
import json
import pandas as pd
import numpy as np

PROJECT_ROOT = "/content/gdrive/MyDrive/CrisisWeave_Project"

LOCATION_RESULTS_DIR = os.path.join(
    PROJECT_ROOT,
    "04_Results",
    "Location_Verification"
)

os.makedirs(
    LOCATION_RESULTS_DIR,
    exist_ok=True
)

print("CrisisWeave Location Verification initialized.")
print("Output directory:", LOCATION_RESULTS_DIR)

CrisisWeave Location Verification initialized.
Output directory: /content/gdrive/MyDrive/CrisisWeave_Project/04_Results/Location_Verification


LOC-02 — Load Dynamic State + Evidence + GIS

In [3]:
# Load Dynamic Disaster State

state_path = os.path.join(
    PROJECT_ROOT,
    "04_Results",
    "Dynamic_State",
    "CrisisWeave_Dynamic_Disaster_State_CW001.json"
)

with open(state_path, "r", encoding="utf-8") as f:
    dynamic_state = json.load(f)


# Load Unified Evidence

evidence_path = os.path.join(
    PROJECT_ROOT,
    "04_Results",
    "Evidence",
    "CrisisWeave_Evidence_CW001_EV001.json"
)

with open(evidence_path, "r", encoding="utf-8") as f:
    evidence_record = json.load(f)


# Load GIS output

gis_path = os.path.join(
    PROJECT_ROOT,
    "04_Results",
    "GIS",
    "CrisisWeave_GIS_CW001.json"
)

with open(gis_path, "r", encoding="utf-8") as f:
    gis_output = json.load(f)


print("Dynamic state loaded.")
print("Evidence loaded.")
print("GIS output loaded.")

Dynamic state loaded.
Evidence loaded.
GIS output loaded.


LOC-03 — Extract verification signals

In [4]:
severity = dynamic_state[
    "current_state"
]["severity"]

risk_level = dynamic_state[
    "current_state"
]["operational_risk"]

humanitarian_category = dynamic_state[
    "humanitarian_state"
]["category"]

humanitarian_support = dynamic_state[
    "humanitarian_state"
]["support_score"]

temporal_trend = dynamic_state[
    "temporal_state"
]["trend"]

escalation_state = dynamic_state[
    "temporal_state"
]["escalation_state"]

gis_score = dynamic_state[
    "geographic_state"
]["critical_context_score"]

response_status = dynamic_state[
    "response_state"
]["status"]

conflict_status = dynamic_state[
    "evidence_quality"
]["conflict_status"]

state_confidence = dynamic_state[
    "evidence_quality"
]["state_confidence"]


gis_context = gis_output[
    "gis_context"
]

print("Severity:", severity)
print("Risk:", risk_level)
print("Humanitarian:", humanitarian_category)
print("Temporal trend:", temporal_trend)
print("GIS context score:", gis_score)
print("Response:", response_status)
print("Conflict:", conflict_status)

Severity: High
Risk: CRITICAL
Humanitarian: requests_or_urgent_needs
Temporal trend: INCREASING
GIS context score: 1.0
Response: En Route
Conflict: AGREEMENT


LOC-04 — Evaluate geographic plausibility

In [8]:
# LOC-04 — Read actual GIS fields safely

gis_context = gis_output["gis_context"]

print("Available GIS fields:")
for key, value in gis_context.items():
    print(f"{key}: {value}")

# LOC-04B — Extract values from the actual GIS structure

nearest_road = gis_context.get("nearest_road", {})
nearest_school = gis_context.get("nearest_school", {})
nearest_hospital = gis_context.get("nearest_hospital", {})
nearest_bridge = gis_context.get("nearest_bridge", {})
nearest_waterbody = gis_context.get("nearest_waterbody", {})

# Distances
nearest_road_distance = nearest_road.get("distance_m")
nearest_school_distance = nearest_school.get("distance_m")
nearest_hospital_distance = nearest_hospital.get("distance_m")
nearest_bridge_distance = nearest_bridge.get("distance_m")
nearest_water_distance = nearest_waterbody.get("distance_m")

# Water relationship
water_relationship = nearest_waterbody.get(
    "relationship"
)

print("Extracted GIS context:")
print("----------------------")
print("Water relationship :", water_relationship)
print("Nearest road       :", nearest_road_distance, "m")
print("Nearest water      :", nearest_water_distance, "m")
print("Nearest bridge     :", nearest_bridge_distance, "m")
print("Nearest school     :", nearest_school_distance, "m")
print("Nearest hospital   :", nearest_hospital_distance, "m")

Available GIS fields:
nearest_road: {'feature_type': 'Road', 'name': 'Syrian Church Road', 'latitude': 1097835.502269, 'longitude': 639258.587067, 'distance_m': 17.2}
nearest_school: {'feature_type': 'School', 'name': 'Kendriya Vidyalaya-Mundamveli Kochi', 'latitude': 1098032.308252, 'longitude': 638927.3767, 'distance_m': 358.26}
nearest_hospital: {'feature_type': 'Hospital', 'name': 'AMJ Hospital', 'latitude': 1097876.28905, 'longitude': 639258.858736, 'distance_m': 38.14}
nearest_bridge: {'feature_type': 'Bridge', 'name': 'Gateway of Kochi Bridge', 'latitude': 1098348.974526, 'longitude': 639212.568086, 'distance_m': 381.73}
nearest_waterbody: {'feature_type': 'Waterbody', 'name': 'Waterbody', 'distance_m': 73.52, 'relationship': 'Near waterbody'}
Extracted GIS context:
----------------------
Water relationship : Near waterbody
Nearest road       : 17.2 m
Nearest water      : 73.52 m
Nearest bridge     : 381.73 m
Nearest school     : 358.26 m
Nearest hospital   : 38.14 m


LOC-05 — Corrected geographic score

In [10]:
# LOC-05 — Corrected geographic plausibility score

geographic_score = 0.0
geographic_reasons = []

# Normalize water relationship
water_relation_normalized = str(
    water_relationship
).strip().lower()


# --------------------------------------------------
# Water proximity
# --------------------------------------------------

if "inside" in water_relation_normalized:
    geographic_score += 0.35

    geographic_reasons.append(
        "Incident point lies inside a mapped water feature."
    )

elif "adjacent" in water_relation_normalized:
    geographic_score += 0.30

    geographic_reasons.append(
        "Incident point is adjacent to mapped water."
    )

elif "near" in water_relation_normalized:
    geographic_score += 0.20

    geographic_reasons.append(
        "Incident point is near a mapped water feature."
    )


# --------------------------------------------------
# Road proximity
# --------------------------------------------------

if (
    nearest_road_distance is not None
    and nearest_road_distance <= 100
):
    geographic_score += 0.20

    geographic_reasons.append(
        f"Incident point is {nearest_road_distance:.2f} m "
        "from a mapped road."
    )


# --------------------------------------------------
# Bridge proximity
# --------------------------------------------------

if (
    nearest_bridge_distance is not None
    and nearest_bridge_distance <= 200
):
    geographic_score += 0.15

    geographic_reasons.append(
        f"Incident point is {nearest_bridge_distance:.2f} m "
        "from a mapped bridge."
    )


# --------------------------------------------------
# School proximity
# --------------------------------------------------

if (
    nearest_school_distance is not None
    and nearest_school_distance <= 500
):
    geographic_score += 0.10

    geographic_reasons.append(
        f"A mapped school is {nearest_school_distance:.2f} m away."
    )


# --------------------------------------------------
# Hospital proximity
# --------------------------------------------------

if (
    nearest_hospital_distance is not None
    and nearest_hospital_distance <= 500
):
    geographic_score += 0.10

    geographic_reasons.append(
        f"A mapped hospital is {nearest_hospital_distance:.2f} m away."
    )


geographic_score = min(
    geographic_score,
    1.0
)

print("Geographic plausibility score:",
      round(geographic_score, 4))

print("\nReasons:")
for reason in geographic_reasons:
    print("-", reason)

Geographic plausibility score: 0.6

Reasons:
- Incident point is near a mapped water feature.
- Incident point is 17.20 m from a mapped road.
- A mapped school is 358.26 m away.
- A mapped hospital is 38.14 m away.


LOC-06 — Combine multi-source verification

In [11]:
# CNN / fused severity support
severity_support = {
    "Low": 0.20,
    "Moderate": 0.60,
    "High": 1.00
}.get(
    severity,
    0.20
)

# Temporal support
temporal_support = {
    "INCREASING": 1.00,
    "STABLE": 0.60,
    "DECREASING": 0.40
}.get(
    temporal_trend,
    0.50
)

# Response support
response_support = {
    "Dispatched": 0.50,
    "En Route": 0.75,
    "Arrived": 1.00,
    "Completed": 0.40
}.get(
    response_status,
    0.50
)

# Conflict adjustment
conflict_adjustment = {
    "AGREEMENT": 1.00,
    "MINOR_CONFLICT": 0.85,
    "MAJOR_CONFLICT": 0.65
}.get(
    conflict_status,
    0.75
)

location_verification_score = (
    0.30 * geographic_score
    + 0.25 * severity_support
    + 0.15 * humanitarian_support
    + 0.15 * temporal_support
    + 0.10 * response_support
    + 0.05 * state_confidence
)

location_verification_score *= conflict_adjustment

location_verification_score = round(
    np.clip(
        location_verification_score,
        0,
        1
    ),
    4
)

print(
    "Multi-source location verification score:",
    location_verification_score
)

Multi-source location verification score: 0.855


LOC-07 — Determine verification status

In [12]:
if location_verification_score >= 0.75:
    verification_status = "VERIFIED"

elif location_verification_score >= 0.50:
    verification_status = "PARTIALLY VERIFIED"

else:
    verification_status = "UNVERIFIED"


print("Location verification status:")
print(verification_status)

Location verification status:
VERIFIED


LOC-08 — Build verification result

In [13]:
location_verification = {
    "incident_id": dynamic_state["incident_id"],

    "location": evidence_record["location"],

    "verification_status": verification_status,

    "verification_score": location_verification_score,

    "components": {
        "geographic_plausibility": round(
            geographic_score,
            4
        ),

        "severity_support": round(
            severity_support,
            4
        ),

        "humanitarian_support": round(
            humanitarian_support,
            4
        ),

        "temporal_support": round(
            temporal_support,
            4
        ),

        "response_support": round(
            response_support,
            4
        ),

        "state_confidence": round(
            state_confidence,
            4
        ),

        "conflict_adjustment": conflict_adjustment
    },

    "reasons": geographic_reasons,

    "interpretation": (
        "Verification indicates whether the reported "
        "location is sufficiently supported by the "
        "available multi-source evidence. It does not "
        "mean that GIS alone confirmed flooding."
    )
}

print(json.dumps(
    location_verification,
    indent=2,
    ensure_ascii=False
))

{
  "incident_id": "CW-001",
  "location": {
    "latitude": 9.929117,
    "longitude": 76.270436,
    "source": "user_map_pin"
  },
  "verification_status": "VERIFIED",
  "verification_score": 0.855,
  "components": {
    "geographic_plausibility": 0.6,
    "severity_support": 1.0,
    "humanitarian_support": 1.0,
    "temporal_support": 1.0,
    "response_support": 0.75,
    "state_confidence": 1.0,
    "conflict_adjustment": 1.0
  },
  "reasons": [
    "Incident point is near a mapped water feature.",
    "Incident point is 17.20 m from a mapped road.",
    "A mapped school is 358.26 m away.",
    "A mapped hospital is 38.14 m away."
  ],
  "interpretation": "Verification indicates whether the reported location is sufficiently supported by the available multi-source evidence. It does not mean that GIS alone confirmed flooding."
}


LOC-09 — Save verification result

In [14]:
location_json_path = os.path.join(
    LOCATION_RESULTS_DIR,
    "CrisisWeave_Location_Verification_CW001.json"
)

with open(
    location_json_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        location_verification,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Location verification saved:")
print(location_json_path)

Location verification saved:
/content/gdrive/MyDrive/CrisisWeave_Project/04_Results/Location_Verification/CrisisWeave_Location_Verification_CW001.json


LOC-10 — Create verification summary

In [15]:
verification_table = pd.DataFrame([{
    "incident_id":
        location_verification["incident_id"],

    "verification_status":
        location_verification["verification_status"],

    "verification_score":
        location_verification["verification_score"],

    "geographic_plausibility":
        location_verification["components"][
            "geographic_plausibility"
        ],

    "severity_support":
        location_verification["components"][
            "severity_support"
        ],

    "humanitarian_support":
        location_verification["components"][
            "humanitarian_support"
        ],

    "temporal_support":
        location_verification["components"][
            "temporal_support"
        ],

    "response_support":
        location_verification["components"][
            "response_support"
        ],

    "conflict_adjustment":
        location_verification["components"][
            "conflict_adjustment"
        ]
}])

verification_csv_path = os.path.join(
    LOCATION_RESULTS_DIR,
    "CrisisWeave_Location_Verification_CW001.csv"
)

verification_table.to_csv(
    verification_csv_path,
    index=False
)

display(verification_table)

print("Summary saved:")
print(verification_csv_path)

,incident_id,verification_status,verification_score,geographic_plausibility,severity_support,humanitarian_support,temporal_support,response_support,conflict_adjustment
0,CW-001,VERIFIED,0.855,0.6,1.0,1.0,1.0,0.75,1.0


Summary saved:
/content/gdrive/MyDrive/CrisisWeave_Project/04_Results/Location_Verification/CrisisWeave_Location_Verification_CW001.csv
